In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-cc82vtij/unsloth_9663164af1a44c228fb0567f965a9cee
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-cc82vtij/unsloth_9663164af1a44c228fb0567f965a9cee
  Resolved https://github.com/unslothai/unsloth.git to commit b3640802253f64117ee228718be7fab32e47aa5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.3 MB/s eta 0:00:0

In [3]:
import torch
import unsloth
import trl
import peft
import bitsandbytes

print("torch:", torch.__version__)
print("unsloth:", unsloth.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
torch: 2.10.0+cu128
unsloth: 2026.5.2
trl: 0.24.0
peft: 0.19.1
CUDA available: True
GPU: Tesla T4
VRAM: 15.64 GB


In [4]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context", split="train")
print(dataset)
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 78577
})
{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


In [5]:
def format_prompt(example):
    return {
        "text": f"""You are a SQL expert. Given a table schema and a question, write the correct SQL query.

### Schema:
{example['context']}

### Question:
{example['question']}

### SQL:
{example['answer']}"""
    }

dataset = dataset.map(format_prompt)
print(dataset[0]['text'])

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

You are a SQL expert. Given a table schema and a question, write the correct SQL query.

### Schema:
CREATE TABLE head (age INTEGER)

### Question:
How many heads of the departments are older than 56 ?

### SQL:
SELECT COUNT(*) FROM head WHERE age > 56


In [6]:
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

print("Train samples:", len(train_dataset))
print("Eval samples:", len(eval_dataset))

Train samples: 74648
Eval samples: 3929


In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B",
    max_seq_length=1024,
    load_in_4bit=True,
)

print("Model loaded.")

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Model loaded.


In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapter applied.")
print(model.print_trainable_parameters())

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapter applied.
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
None


In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=500,
        save_steps=500,
        output_dir="sql-model",
        report_to="none",
    ),
)

print("Trainer ready.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/74648 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3929 [00:00<?, ? examples/s]

Trainer ready.


In [11]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 74,648 | Num Epochs = 1 | Total steps = 9,331
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
500,0.713674,0.696036
1000,0.674893,0.667715
1500,0.658772,0.652082
2000,0.656834,0.641613
2500,0.640594,0.632194
3000,0.642934,0.624760
3500,0.629063,0.619012
4000,0.641680,0.613252
4500,0.616117,0.607666
5000,0.610601,0.603087


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=9331, training_loss=0.6252202526726609, metrics={'train_runtime': 17152.0282, 'train_samples_per_second': 4.352, 'train_steps_per_second': 0.544, 'total_flos': 5.872048360936243e+16, 'train_loss': 0.6252202526726609, 'epoch': 1.0})

In [13]:
model.save_pretrained("sql-finetuned")
tokenizer.save_pretrained("sql-finetuned")
print("Model saved.")

Unsloth: Restored added_tokens_decoder metadata in sql-finetuned/tokenizer_config.json.


Model saved.


In [18]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def generate_sql(question, schema):
    prompt = f"""You are a SQL expert. Given a table schema and a question, write the correct SQL query.

### Schema:
{schema}

### Question:
{question}

### SQL:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        repetition_penalty=1.5,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql = response.split("### SQL:")[-1].strip()
    # Cut at first newline to stop over-generation
    return sql.split("\n")[0].strip()

# Test questions
schema = "CREATE TABLE employees (id INTEGER, name TEXT, salary INTEGER, department TEXT)"

questions = [
    "How many employees are there?",
    "What is the average salary?",
    "List all employees in the Engineering department.",
    "Who has the highest salary?",
    "How many employees are in each department?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"SQL: {generate_sql(q, schema)}")
    print()

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How many employees are there?


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL: SELECT COUNT(*) FROM employees WHERE id > 0 AND NOT NAME = 'John' OR SALARY < -156982347

Q: What is the average salary?


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL: SELECT AVG(salary) FROM employees WHERE NOT id IN (

Q: List all employees in the Engineering department.


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL: SELECT id FROM employees WHERE salary > 105689 AND NOT Department = 'Engineering' UNION SELECT * from Employees where Name LIKE '%engineering%' OR Salary < %salary% ORDER BY ID DESC LIMIT $limit$;

Q: Who has the highest salary?


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL: SELECT id FROM employees ORDER BY salary DESC LIMIT 1

Q: How many employees are in each department?
SQL: SELECT COUNT(*), SUM(salary) FROM (

